# Introduction
This notebook demonstrates the fine-tuning of Phi-4 for the AI Mathematical Olympiad 2 competition (Link to Kaggle competition page: https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2).

Phi-4 is Microsoft’s latest small language model, designed specifically for complex reasoning tasks. In the introduction article, "Introducing Phi-4: Microsoft’s Newest Small Language Model Specializing in Complex Reasoning," Phi-4 is highlighted for its exceptional performance on math problems, with impressive benchmark results on math competition challenges.

Phi-4 is a small model, but still large enough to present challenges in a Kaggle notebook environment. This notebook refers to Unsloth's "Phi-4 Fine-tuning" notebook and demonstrates how to fine-tune Phi-4 within the Kaggle notebook environment. The fine-tuning process uses the Math Problems IMO dataset, which contains 100,000 pairs of mathematical problems and their corresponding solutions.

# Preparing model and tokenizer

In [1]:
%%capture
!pip install unsloth

In [6]:
# Speed up training during debugging
# If DEBUG is True, sample 100 rows from the training data and limit max_steps to 10 for faster iteration.
# If DEBUG is False, use the entire dataset and set max_steps to None for full training.
DEBUG = True

if DEBUG:
    # Sample a small subset of the data for quick debugging
    dfTrain = dfTrain.head(100)
    max_steps = 10  # Limit the number of steps for debugging purposes
else:
    max_steps = None  # Use the entire dataset for full training

In [2]:
import random
import os
import numpy as np
from transformers import set_seed

# Set a fixed seed for reproducibility across various libraries
SEED = 40

# Set the seed for random number generation in Python's random module
random.seed(SEED)

# Set the seed for hash randomization in Python (affects os and related libraries)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Set the seed for numpy random number generation
np.random.seed(SEED)

# Set the seed for Hugging Face transformers
set_seed(SEED)

In [3]:
from unsloth import FastLanguageModel  # FastLanguageModel for LLMs
import torch

# Set the maximum sequence length for the model
max_seq_length = 2048  # You can choose any length. RoPE Scaling is supported internally!

# Set whether to use 4-bit quantization for reduced memory usage (Can be set to False)
load_in_4bit = True

# List of pre-quantized 4-bit models for faster downloads and to avoid OOM (Out of Memory) errors
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",  # Llama-3.1 (2x faster)
    "unsloth/Mistral-Small-Instruct-2409",  # Mistral 22b (2x faster)
    "unsloth/Phi-4",  # Phi-4 (2x faster)
    "unsloth/Phi-4-unsloth-bnb-4bit",  # Phi-4 with Unsloth Dynamic 4-bit Quantization
    "unsloth/gemma-2-9b-bnb-4bit",  # Gemma (2x faster)
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # Qwen 2.5 (2x faster)
    "unsloth/Llama-3.2-1B-bnb-4bit",  # Llama 3.2 1B model (new)
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",  # Llama 3.2 1B Instruction-tuned model
    "unsloth/Llama-3.2-3B-bnb-4bit",  # Llama 3.2 3B model
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",  # Llama 3.2 3B Instruction-tuned model
]  # More models available at https://docs.unsloth.ai/get-started/all-our-models

# Load the pre-trained model with the specified settings
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=fourbit_models[2],  # Phi-4 model (can be changed to any model from the list)
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,  # Use 4-bit quantization if enabled
)


<ipython-input-3-d548512640f1>:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel  # FastLanguageModel for LLMs


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/160k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.61M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/917k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

# Addidng LORA adapter

In [4]:
# Apply Parameter-Efficient Fine-Tuning (PEFT) to the model using LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Rank for LoRA, choose any value > 0. Recommended: 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention-related modules
        "gate_proj", "up_proj", "down_proj"  # LoRA-specific projection modules
    ],
    lora_alpha = 16,  # Scaling factor for LoRA (often between 8 and 16)
    lora_dropout = 0,  # Dropout for LoRA, 0 is optimized for performance
    bias = "none",  # Optimization for biases; "none" is usually the best choice
    use_gradient_checkpointing = "unsloth",  # Use gradient checkpointing for memory optimization
    random_state = 3407,  # Set a random seed for reproducibility
    use_rslora = False,  # Rank Stabilized LoRA (set to False if not used)
    loftq_config = None,  # Optional configuration for LoftQ, leave as None if unused
)

Unsloth 2025.3.19 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


# Prepare data for finetuning
The dataset used for fine-tuning is the Math Problems IMO dataset, which contains 100,000 pairs of mathematical problems and their corresponding solutions. For fine-tuning Phi-4, the dataset is first converted to ShareGPT style, and then transformed into Hugging Face's standard multi-turn conversation format. This process enables the dataset to be rendered as multi-turn conversations, making it suitable for training a conversational AI model.

In [7]:
from datasets import Dataset
import pandas as pd

# Load the Math Problems IMO dataset in Parquet format using pandas
dfTrain = pd.read_parquet(f"/kaggle/input/math-problems-imo/math_problems.parquet")

# If in debugging mode, limit the dataset to the first 100 rows for faster iteration
if DEBUG:
    dfTrain = dfTrain.head(100)

# Convert the DataFrame into a Hugging Face Dataset for further processing
train_dataset = Dataset.from_pandas(dfTrain)

In [8]:
# Print the total number of examples available for fine-tuning
print(f"The number of examples for fine-tuning is {dfTrain.shape[0]} \n")

# Display an example pair of a problem and its solution from the dataset
print(f"Example pair of problem and solution: \n")
print(f"Problem: {dfTrain['problem'][2]} \n")
print(f"Solution: {dfTrain['solution'][2]} \n")

The number of examples for fine-tuning is 100 

Example pair of problem and solution: 

Problem: Eduardo is a teacher. He taught 3 classes last week while his colleague Frankie taught double what Eduardo teaches. How many classes did Eduardo and Frankie teach in total? 

Solution: To solve the problem, follow these steps:

1. Determine the number of classes Eduardo taught.
2. Calculate the number of classes Frankie taught, which is double the number of classes Eduardo taught.
3. Compute the total number of classes taught by both Eduardo and Frankie.

First, identify the number of classes Eduardo taught (which is given as 3).

Next, calculate the number of classes Frankie taught:

\[
\text{Frankie's classes} = 2 \times \text{Eduardo's classes}
\]

Finally, calculate the total number of classes taught by both:

\[
\text{Total classes} = \text{Eduardo's classes} + \text{Frankie's classes}
\]

Let's go ahead and compute these in Python.
```python
# Step 1: Number of classes Eduardo taught


In [9]:
# Format the DataFrame into ShareGPT style by creating conversations with 'human' and 'gpt' roles
dfTrain['conversations'] = dfTrain.apply(
    lambda row: [
        {"from": "human", "value": row["problem"]},  # Human prompt (math problem)
        {"from": "gpt", "value": row["solution"]}    # GPT response (solution)
    ], axis=1
)

# Create a Hugging Face Dataset from the formatted DataFrame, containing only the 'conversations' column
dataset = Dataset.from_pandas(dfTrain[['conversations']])

# Verify the dataset structure and content
print(dataset)

Dataset({
    features: ['conversations'],
    num_rows: 100
})


In [10]:
# Print an example from the dataset to verify its structure and content
print(f"Example of dataset: \n {dataset[1]} \n")

Example of dataset: 
 {'conversations': [{'from': 'human', 'value': 'Find all integer solutions to the equation \\(3x - 12y = 7\\).'}, {'from': 'gpt', 'value': '\nTo determine whether there are any integer solutions to the equation \\(3x - 12y = 7\\), we need to consider the divisibility properties of the left-hand side and the right-hand side of the equation.\n\n1. **Analyzing Divisibility:**\n   - The left-hand side of the equation is \\(3x - 12y\\).\n   - Notice that both terms, \\(3x\\) and \\(12y\\), are divisible by 3.\n   - Therefore, the entire expression \\(3x - 12y\\) is divisible by 3.\n\n2. **Examining the Right-Hand Side:**\n   - The right-hand side of the equation is \\(7\\).\n   - The number \\(7\\) is **not divisible** by 3.\n\n3. **Conclusion from Divisibility:**\n   To have integer solutions \\((x, y)\\) for the equation \\(3x - 12y = 7\\), both sides of the equation must be congruent modulo 3.\n\n   - If \\(a \\equiv b \\pmod{m}\\), then both \\(a\\) and \\(b\\) must

Unsloth provides the get_chat_template function to retrieve the appropriate chat template. The following code defines a function that utilizes it specifically for Phi-4.

In [11]:
from unsloth.chat_templates import get_chat_template

# Load the Phi-4 chat template using Unsloth's get_chat_template function
tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",  # Specify the Phi-4 template
)

# Define a function to format prompts using the chat template
def format_prompts(examples):
    # Extract conversations from the input examples
    convos = examples["conversations"]
    
    # Apply the Phi-4 chat template to each conversation without tokenizing or adding a generation prompt
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    
    # Return the formatted texts as a dictionary
    return {"text": texts}

In [12]:
# Convert ShareGPT style to HuggingFace's standard multi-turn conversation format
from unsloth.chat_templates import standardize_sharegpt

# Standardize the dataset from ShareGPT format to HuggingFace multi-turn format
dataset = standardize_sharegpt(dataset)

# Apply the formatting function to the dataset in batches
dataset = dataset.map(
    format_prompts,  # Use the format_prompts function defined earlier
    batched=True,    # Process the dataset in batches for efficiency
)

Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
# Print an example of the dataset after conversion to HuggingFace's generic format
print(f"Example of converted dataset into HuggingFace's generic format: \n")

# Print the 'conversations' and 'text' fields of the third example in the dataset
print(f"Conversations: {dataset[2]['conversations']}\n")
print(f"Text: {dataset[2]['text']}\n")

Example of converted dataset into HuggingFace's generic format: 

Conversations: [{'content': 'Eduardo is a teacher. He taught 3 classes last week while his colleague Frankie taught double what Eduardo teaches. How many classes did Eduardo and Frankie teach in total?', 'role': 'user'}, {'content': "To solve the problem, follow these steps:\n\n1. Determine the number of classes Eduardo taught.\n2. Calculate the number of classes Frankie taught, which is double the number of classes Eduardo taught.\n3. Compute the total number of classes taught by both Eduardo and Frankie.\n\nFirst, identify the number of classes Eduardo taught (which is given as 3).\n\nNext, calculate the number of classes Frankie taught:\n\n\\[\n\\text{Frankie's classes} = 2 \\times \\text{Eduardo's classes}\n\\]\n\nFinally, calculate the total number of classes taught by both:\n\n\\[\n\\text{Total classes} = \\text{Eduardo's classes} + \\text{Frankie's classes}\n\\]\n\nLet's go ahead and compute these in Python.\n```p

# Finetuning
To train the model, we will use Hugging Face's SFTTrainer. Due to the limitations of the notebook environment, max_steps is set to a fixed value. If we want to perform full training, we have to set num_train_epochs=1 and set max_steps=None to remove the step limit.

In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

# Set max_steps for debugging or full training
if DEBUG:
    max_steps = 10  # For debugging, limit steps to 10
else:
    max_steps = None  # No step limit for full training

# Initialize the SFTTrainer for training the model
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",  # Specify the field to use from the dataset
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),  # Collator for padding and batching
    dataset_num_proc=2,  # Number of processes for dataset loading
    packing=False,  # Disable packing to avoid potential memory issues for longer sequences

    # Training arguments configuration
    args=TrainingArguments(
        per_device_train_batch_size=2,  # Batch size per device (GPU)
        gradient_accumulation_steps=4,  # Accumulate gradients over multiple steps
        warmup_steps=5,  # Steps to perform learning rate warmup
        num_train_epochs=1,  # Set to 1 for a full training run
        max_steps=max_steps,  # Limit the number of training steps (None for full training)
        learning_rate=2e-4,  # Learning rate for training
        fp16=not is_bfloat16_supported(),  # Use FP16 if bfloat16 is not supported
        bf16=is_bfloat16_supported(),  # Use bfloat16 if supported
        logging_steps=1,  # Log every step
        optim="adamw_8bit",  # Optimizer with 8-bit precision
        weight_decay=0.01,  # Weight decay for regularization
        lr_scheduler_type="linear",  # Linear learning rate scheduler
        seed=3407,  # Random seed for reproducibility
        output_dir="outputs",  # Directory to save model outputs
        report_to="none",  # Disable reporting (e.g., to WandB)
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

Unsloth's train_on_responses_only method allows training exclusively on the assistant's outputs, while ignoring the loss from the user's inputs.

In [15]:
from unsloth.chat_templates import train_on_responses_only

# Apply 'train_on_responses_only' to focus training on the assistant's responses
trainer = train_on_responses_only(
    trainer,  # Pass the existing trainer object
    instruction_part="<|im_start|>user<|im_sep|>",  # Marker for user input in the conversation
    response_part="<|im_start|>assistant<|im_sep|>",  # Marker for assistant output in the conversation
)

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Checking masking

In [16]:
decoded_text = tokenizer.decode(trainer.train_dataset[5]["input_ids"])

print(decoded_text)

<|im_start|>user<|im_sep|>Given that point $P$ is on the right branch of the hyperbola $\frac{x^2}{a^2} - \frac{y^2}{b^2} = 1 (a > 0, b > 0)$, and $F_1$, $F_2$ are the left and right foci of the hyperbola, respectively. It is given that $(\overrightarrow{OP} + \overrightarrow{OF_2}) \cdot \overrightarrow{F_2P} = 0$ (where $O$ is the origin), and $|\overrightarrow{PF_1}| = \sqrt{3}|\overrightarrow{PF_2}|$. The eccentricity of the hyperbola is
A: $\frac{\sqrt{6}+1}{2}$
B: $\sqrt{6}+1$
C: $\frac{\sqrt{3}+1}{2}$
D: $\sqrt{3}+1$<|im_end|><|im_start|>assistant<|im_sep|>Since $(\overrightarrow{OP} + \overrightarrow{OF_2}) \cdot \overrightarrow{F_2P} = 0$ (where $O$ is the origin),  
it follows that $|\overrightarrow{OP}| = |\overrightarrow{OF_2}|$, thus $|\overrightarrow{OP}| = |\overrightarrow{OF_2}| = |\overrightarrow{OF_1}| = c$,  
therefore $\angle F_1PF_2 = 90^{\circ}$.  
Let $|PF_2| = x$, then $|PF_1| = \sqrt{3}x$,  
$\sqrt{3}x - x = 2a$, solving this gives $|PF_1| = (\sqrt{3}+3)a, |PF_

In [17]:
# Retrieve the token ID for a space character (without special tokens)
space = tokenizer(" ", add_special_tokens=False).input_ids[0]

# Decode the labels, replacing -100 with a space token
decoded_labels = tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

print(decoded_labels)

                                                                                                                                                                                                                       Since $(\overrightarrow{OP} + \overrightarrow{OF_2}) \cdot \overrightarrow{F_2P} = 0$ (where $O$ is the origin),  
it follows that $|\overrightarrow{OP}| = |\overrightarrow{OF_2}|$, thus $|\overrightarrow{OP}| = |\overrightarrow{OF_2}| = |\overrightarrow{OF_1}| = c$,  
therefore $\angle F_1PF_2 = 90^{\circ}$.  
Let $|PF_2| = x$, then $|PF_1| = \sqrt{3}x$,  
$\sqrt{3}x - x = 2a$, solving this gives $|PF_1| = (\sqrt{3}+3)a, |PF_2| = (\sqrt{3}+1)a$,  
thus $c = \frac{\sqrt{(\sqrt{3}+3)^2a^2 + (\sqrt{3}+1)^2a^2}}{2} = (\sqrt{3}+1)a$,  
therefore $e = \frac{c}{a} = \sqrt{3}+1$.  
Hence, the correct choice is $\boxed{D}$.  
First, from $(\overrightarrow{OP} + \overrightarrow{OF_2}) \cdot \overrightarrow{F_2P} = 0$, we deduce that $\angle F_1PF_2 = 90^{\circ}$. Then, using $|\overr

In [18]:
# @title Show current GPU memory statistics

# Get properties of the GPU (assuming GPU 0 is being used)
gpu_stats = torch.cuda.get_device_properties(0)

# Get the maximum memory reserved by the model/process, in GB
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# Get the total memory of the GPU, in GB
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

# Print out the GPU name and the memory stats
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"Memory reserved at start: {start_gpu_memory} GB.")

GPU = Tesla T4. Max memory = 14.741 GB.
Memory reserved at start: 9.967 GB.


In [19]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 2 | Total steps = 10
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 65,536,000/4,000,000,000 (1.64% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.576700
2,0.532300
3,0.605700
4,0.565000
5,0.556700
6,0.503900
7,0.404100
8,0.430800
9,0.412000
10,0.445900


In [20]:
# @title Show final memory and time statistics

# Calculate the memory used during training (in GB)
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# Calculate the memory used specifically for LoRA fine-tuning
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)

# Calculate the percentage of GPU memory used relative to the total memory
used_percentage = round(used_memory / max_memory * 100, 3)

# Calculate the percentage of memory used for LoRA fine-tuning relative to the total GPU memory
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

# Print out the training time and memory stats
print(f"Total training time: {trainer_stats.metrics['train_runtime']} seconds.")
print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes.")

# Print the memory stats
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory usage as a percentage of total memory = {used_percentage}%.")
print(f"Peak reserved memory for training as a percentage of total memory = {lora_percentage}%.")

Total training time: 730.386 seconds.
Training time: 12.17 minutes.
Peak reserved memory = 13.715 GB.
Peak reserved memory for training = 3.748 GB.
Peak reserved memory usage as a percentage of total memory = 93.04%.
Peak reserved memory for training as a percentage of total memory = 25.426%.


# Test inference
We test the finetuned model with sample from train data.

In [21]:
dfTrain

,problem,solution,conversations
0,Factor \( t^2 - 144 \).,1. Observe the expression \( t^2 - 144 \). It ...,"[{'from': 'human', 'value': 'Factor \( t^2 - 1..."
1,Find all integer solutions to the equation \(3...,\nTo determine whether there are any integer s...,"[{'from': 'human', 'value': 'Find all integer ..."
2,Eduardo is a teacher. He taught 3 classes last...,"To solve the problem, follow these steps:\n\n1...","[{'from': 'human', 'value': 'Eduardo is a teac..."
3,Free Christmas decorations are being given out...,Each box contains:\n- 4 pieces of tinsel\n- 1 ...,"[{'from': 'human', 'value': 'Free Christmas de..."
4,According to a report by People's Daily on May...,"The form of scientific notation is $a×10^n$, w...","[{'from': 'human', 'value': 'According to a re..."
...,...,...,...
95,Given is a circle $\omega$ and a line $\ell...,1. **Identify the given elements and their rel...,"[{'from': 'human', 'value': 'Given is a circle..."
96,Given that the graph of the function $f(x) = \...,Since the graph of $y = x + \frac{1}{x}$ is sy...,"[{'from': 'human', 'value': 'Given that the gr..."
97,"Tatuya, Ivanna, and Dorothy took a quiz togeth...","Let's denote Dorothy's score as D, Ivanna's sc...","[{'from': 'human', 'value': 'Tatuya, Ivanna, a..."
98,"Villages $A$, $B$, and $C$ are located at the ...","### Problem\nVillages \( A, B \), and \( C \) ...","[{'from': 'human', 'value': 'Villages $A$, $B$..."


In [22]:
# Clear GPU memory before using
torch.cuda.empty_cache()

# Get a problem and its true solution from the training data
problem = dfTrain["problem"][0]
true_solution = dfTrain["solution"][0]

# Import the get_chat_template function from Unsloth
from unsloth.chat_templates import get_chat_template

# Get the Phi-4 chat template and apply it to the tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "phi-4",  # Choose Phi-4 template for the tokenizer
)

# Enable native 2x faster inference for the model
FastLanguageModel.for_inference(model)

# Prepare the input messages (problem) for the model
messages = [
    {"role": "user", "content": problem},  # User message containing the problem
]

# Apply the chat template to the input messages and prepare the tensors for model input
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,  # Add generation prompt for the model
    return_tensors = "pt",  # Return in PyTorch tensor format
).to("cuda")  # Move the tensors to GPU

# Generate the model's response (solution) based on the problem
outputs = model.generate(
    input_ids = inputs, 
    max_new_tokens = 1000,  # Maximum number of tokens to generate
    use_cache = True,  # Use cached key-value states for faster inference
    temperature = 1.5,  # Controls randomness of predictions (higher is more random)
    min_p = 0.1,  # Minimum probability threshold for token selection
)

# Decode and print the model's output (solution)
print(f"Model output: {tokenizer.batch_decode(outputs)} /n")

# Divider line
print("-" * 50)

# Print the true solution for comparison
print(f"Example solution: {true_solution} /n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model output: ['<|im_start|>user<|im_sep|>Factor \\( t^2 - 144 \\).<|im_end|><|im_start|>assistant<|im_sep|>To factor the expression \\( t^2 - 144 \\), we recognize that it is a difference of squares. The general form for the difference of squares is:\n\n\\[\na^2 - b^2 = (a - b)(a + b)\n\\]\n\nIn this case, we identify \\( a \\) and \\( b \\) such that:\n\n\\[\na^2 = t^2 \\quad \\text{and} \\quad b^2 = 144\n\\]\n\nFrom \\( a^2 = t^2 \\), we have \\( a = t \\).\n\nFrom \\( b^2 = 144 \\), we have \\( b = 12 \\) because \\( 12^2 = 144 \\).\n\nSubstituting \\( a = t \\) and \\( b = 12 \\) into the difference of squares formula, we get:\n\n\\[\nt^2 - 144 = (t - 12)(t + 12)\n\\]\n\nThus, the factored form of \\( t^2 - 144 \\) is:\n\n\\[\n\\boxed{(t - 12)(t + 12)}\n\\]<|im_end|>'] /n
--------------------------------------------------
Example solution: 1. Observe the expression \( t^2 - 144 \). It resembles the difference of squares formula \( a^2 - b^2 = (a-b)(a+b) \).
2. Identify \( a = t \)

# Saving and loading the fine-tuned model

In [23]:
# Save LoRA adapter
if True:
    model.save_pretrained("lora_model")  # Save model locally
    tokenizer.save_pretrained("lora_model")  # Save tokenizer locally
    # Uncomment below lines to save to Hugging Face Hub
    # model.push_to_hub("your_name/lora_model", token="...")  # Save model to Hugging Face Hub
    # tokenizer.push_to_hub("your_name/lora_model", token="...")  # Save tokenizer to Hugging Face Hub

# Load LoRA adapter
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="lora_model",  # Load the saved model
        max_seq_length=max_seq_length,  # Same configuration as during training
        dtype=dtype,  # Data type of the model (e.g., float16)
        load_in_4bit=load_in_4bit,  # Load model in 4bit if applicable
    )
    FastLanguageModel.for_inference(model)  # Enable faster inference (2x faster)

In [24]:
# Save the full model as merged to 16-bit
if False:
    model.save_pretrained_merged("AIMO2_06_phi-4_finetune", tokenizer, save_method="merged_16bit")  # Save model locally as 16-bit merged
    # Uncomment below line to push the model to Hugging Face Hub
    # model.push_to_hub_merged("hf/model", tokenizer, save_method="merged_16bit", token="")  # Push model to Hugging Face Hub as 16-bit merged